# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/6(화) 오전 · LLM · 프롬프트 — 실습</mark>

지금까지 만든 룰은 우리가 정한 조건에 맞는 로그만 찾았습니다.

오늘은 LLM 에게 로그를 읽히고 **무슨 일인지 문장으로** 받습니다. 호출하고 → 지시를 다듬고 → 답을 JSON 으로 받습니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 이 파일을 놓는 자리

1. 단톡방에서 받은 이 파일을 `security-agent-toolkit` 폴더 안의 **`agent_core` 폴더**에 넣습니다. `agent_core` 폴더가 없으면 만듭니다.
2. VS Code 에서 `security-agent-toolkit` 폴더를 열고, 왼쪽 목록의 `agent_core` 에서 이 파일을 엽니다.
3. 오른쪽 위 **커널 선택(Select Kernel)** 을 눌러 **각자 만든 가상환경**을 고릅니다. 목록에서 이름에 `.venv` 처럼 가상환경 이름이 붙은 항목입니다. 그냥 `Python 3.x` 만 적힌 항목이 아닙니다.
4. 셀 실행은 **Shift + Enter** 입니다.
5. 「시작하기」 맨 아래의 **위치 확인 셀**을 실행합니다. `준비 완료` 가 나와야 합니다. 둘째 줄의 경로에 가상환경 폴더 이름이 보이는지도 봅니다.

### ⚠ 중요 · 무료 키는 1분에 15번까지만 호출할 수 있습니다

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px; border-left:6px solid #e65100">
<b>무료 Gemini API 키의 한도는 1분에 15번입니다.</b><br>
1. 한도를 넘으면 상태 코드 <code>429</code> 가 반환되고 답을 받지 못합니다.<br>
2. <code>429</code> 가 나오면 <b>1분 기다린 뒤</b> 같은 셀을 다시 실행합니다. 키가 고장 난 것이 아닙니다.<br>
3. 같은 셀을 연달아 여러 번 실행하지 않습니다. LLM 을 세 번 호출하는 문제는 한 번 실행에 세 번을 씁니다.<br>
4. 한도에 걸리면 <code>[한도 초과] 429</code> 나 <code>[LLM 호출 실패] 상태 코드 429</code> 가 먼저 보이고, 그 아래에 <code>KeyError: 'choices'</code> 나 <code>TypeError: 'NoneType' object is not subscriptable</code> 같은 에러가 이어질 수 있습니다. <b>코드가 틀린 것이 아닙니다.</b> 1분 기다린 뒤 같은 셀을 다시 실행합니다.<br>
5. LLM 의 답 문장과 위험도는 실행할 때마다 달라질 수 있습니다. 🎯 와 글자가 같지 않아도 <b>모양이 같으면</b> 맞습니다.
</div>

### 0.2 오늘 오전의 순서

| 교시 | 무엇 |
|---|---|
| 1교시 | 아침 과제 5 — Gemini API 키 발급 · `.env` · `.gitignore` |
| 2교시 | LLM 을 API 로 호출한다 — 주소 · 헤더 · 본문 · 응답에서 답 꺼내기 |
| 3교시 | 프롬프트 — 역할 · 지시 · 출력 형식 · 예시 |
| 4교시 | JSON 으로 받기 — `llm_client.py` |

### 0.3 막혔을 때

1. 문제 앞에 **🐍 문법 상자**가 있으면 먼저 읽습니다. 그 문제에서 쓰는 문법입니다.
2. 문제 아래 **💡 힌트**를 순서대로 따라 합니다.
3. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다.
4. 도전 문제는 안 풀고 넘어가도 됩니다.

폴더는 이런 모양이 됩니다.

```
security-agent-toolkit/
  .env                ← 아침 과제에서 만든 키 파일
  .gitignore
  agent_core/
    261006_am_llm_prompt.ipynb   ← 이 파일
    llm_client.py                 ← 4교시에 여기에 만듭니다
  docs/               ← 그날 기록
```

### 0.4 문제를 푸는 법

1. 코드 셀의 번호 주석이 **무엇을 쓸지** 알려 줍니다. 주석 바로 아래 빈 줄에 코드를 씁니다.
2. 주석에 적힌 **변수 이름을 그대로** 씁니다. 아래 `print` 줄이 그 이름을 쓰기 때문입니다.
3. 앞 문제에서 쓴 줄은 뒤 문제에 **미리 채워져 있습니다.** 그 문제에서 새로 배우는 줄만 씁니다.
4. 미리 채워 둔 줄은 앞 날짜에 배운 것입니다. 고치지 않습니다.
5. 2교시 중간부터는 셀마다 되풀이되는 준비 코드를 **함수 `post_to_llm` 하나**로 줄여 씁니다. `2.4` 에서 한 줄씩 설명합니다.

In [1]:
%pip install requests


Note: you may need to restart the kernel to use updated packages.


In [2]:
# 위치 확인 셀 — 그대로 실행합니다. 뒤의 셀이 이 셀의 ENV_PATH 를 씁니다
import os                                                     # 폴더 · 파일을 다루는 도구
import sys                                                    # 지금 실행 중인 파이썬의 정보를 주는 도구


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                          # 지금 폴더에서 다섯 칸 위까지 찾습니다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 돌려준다
        folder = os.path.dirname(folder)        # 한 칸 위 폴더
    return None


ENV_PATH = find_env()                                         # 찾은 .env 경로. 못 찾으면 None

print("노트북이 실행되는 폴더:", os.getcwd())
print("지금 쓰는 파이썬     :", sys.executable)

if ENV_PATH:
    print("키 파일(.env)         :", ENV_PATH)
    print("준비 완료")
else:
    print(".env 를 찾지 못했습니다")
    print("1) 이 노트북을 security-agent-toolkit 폴더 안의 agent_core 폴더로 옮긴 뒤 다시 엽니다")
    print("2) .env 가 security-agent-toolkit 폴더에 있는지 봅니다 (아침 과제 5)")

노트북이 실행되는 폴더: c:\Users\User\Documents\security-agent-toolkit\agent_core
지금 쓰는 파이썬     : c:\Users\User\Documents\security-agent-toolkit\.venv\Scripts\python.exe
키 파일(.env)         : c:\Users\User\Documents\security-agent-toolkit\agent_core\.env
준비 완료


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">2교시 (10:00–10:50) · LLM 을 API 로 호출한다</mark>

아래 셀을 먼저 실행합니다. `requests` 를 설치합니다.

`.env` 는 `security-agent-toolkit` 폴더에 있습니다. 「시작하기」의 **위치 확인 셀**이 지금 폴더에서 위쪽 폴더로 올라가며 `.env` 를 찾아, 그 경로를 변수 **`ENV_PATH`** 에 담아 두었습니다. 코드에서는 `open(ENV_PATH, …)` 로 읽습니다.

⚠ `NameError: name 'ENV_PATH' is not defined` 가 나오면 위치 확인 셀을 실행하지 않은 것입니다. 「시작하기」로 올라가 그 셀을 실행합니다.

In [3]:
%pip install -q requests

Note: you may need to restart the kernel to use updated packages.


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 상관없습니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **토큰 (token)** | LLM 이 글을 어떤 단위로 잘라서 세는가 |
| **환각 (hallucination)** | LLM 이 틀린 답을 낼 때 어떤 모양으로 틀리는가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 토큰 →
- 환각 →

적어 둔 것은 2교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.

---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2 · LLM 을 API 로 호출한다</mark>

### 왜 필요한가

1. 지금까지 만든 룰은 **우리가 정한 조건**에 맞는 로그만 찾았습니다. 조건에 없는 로그는 그대로 지나갑니다.
2. LLM 은 로그 한 줄을 읽고 무슨 일인지 **문장으로 설명**합니다. 사람이 한 줄씩 읽던 일을 대신합니다.
3. LLM 을 쓰는 방법은 새로 배우지 않습니다. 10/2 에 배운 `requests.post` 로 요청을 보내고 JSON 응답을 받습니다.

### 2교시에 사용하는 용어

| 용어 (읽는 법) | 뜻 | 2교시의 예 |
|---|---|---|
| LLM (엘엘엠) | 다음에 올 토큰을 확률로 예측하는 언어 모델 | `"model": "gemini-3.5-flash-lite"` |
| 토큰 (token) | 모델이 글을 자르는 최소 단위. 사용량과 한도를 토큰으로 센다 | `"total_tokens": 50` |
| 환각 (hallucination · 할루시네이션) | 사실이 아닌 것을 그럴듯하게 지어내는 것 | LLM `SK텔레콤` ↔ 조회 API `SamsungSDS Inc.` |

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.1 LLM 의 응답은 딕셔너리로 옵니다</mark>

「한 문장으로 답하세요. 피싱이란 무엇입니까?」 라고 물었을 때 실제로 받은 응답입니다. 오늘 쓰지 않는 키는 줄였습니다.

```python
data = {
    "choices": [
        {"message": {"role": "assistant",
                     "content": "피싱이란 신뢰할 수 있는 기관이나 사람으로 위장하여 타인의 개인정보나 금융정보를 속여 빼내는 전자 금융 사기 수법입니다."}}
    ],
    "model": "gemini-3.5-flash-lite",
    "usage": {"prompt_tokens": 15, "completion_tokens": 35, "total_tokens": 50}
}

print(data["choices"][0]["message"]["content"])
```

1. 답 문장은 `choices` → `0` → `message` → `content` 순서로 들어가야 나옵니다.
2. `choices` 는 **리스트**라서 숫자 `0` 으로 꺼냅니다. `message` 와 `content` 는 **딕셔너리의 키**라서 이름으로 꺼냅니다.
3. `usage` 에는 토큰 수가 있습니다. 질문이 15토큰, 답이 35토큰, 합계가 50토큰입니다.

9/28 오후에 배운 중첩 자료와 같은 방법입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
data = {
    "choices": [
        {"message": {"role": "assistant",
                     "content": "포트 스캔이란 컴퓨터 네트워크의 취약점을 찾기 위해 특정 서버의 열려 있는 포트와 활성화된 서비스 탐색을 시도하는 해킹 또는 보안 진단 기법입니다."}}
    ],
    "usage": {"prompt_tokens": 17, "completion_tokens": 43, "total_tokens": 60}
}

print(data["choices"][0]["message"]["role"])
```

막히면 바로 위 `2.1 LLM 의 응답은 딕셔너리로 옵니다` 설명을 다시 봅니다.

In [4]:
data = {
    "choices": [
        {"message": {"role": "assistant",
                     "content": "포트 스캔이란 컴퓨터 네트워크의 취약점을 찾기 위해 특정 서버의 열려 있는 포트와 활성화된 서비스 탐색을 시도하는 해킹 또는 보안 진단 기법입니다."}}
    ],
    "usage": {"prompt_tokens": 17, "completion_tokens": 43, "total_tokens": 60}
}

print(data["choices"][0]["message"]["role"])

assistant


✅ `assistant`

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.2 호출에는 주소 · 헤더 · 본문이 필요합니다</mark>

9/30 과 10/2 에 쓴 `requests` 그대로입니다. 넣는 값만 다릅니다.

```python
import requests

api_key = "발급받은_키"      # 실제로는 .env 에서 읽습니다

url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
body = {
    "model": "gemini-3.5-flash-lite",
    "messages": [{"role": "user", "content": "한 문장으로 답하세요. 피싱이란 무엇입니까?"}]
}

response = requests.post(url, headers=headers, json=body, timeout=30)
print(response.status_code)          # 200
```

1. **주소** — LLM 서비스가 요청을 받는 곳입니다.
2. **헤더** — `Authorization` 에 `Bearer` 와 공백 한 칸, 그 뒤에 키를 붙입니다. 9/29 오후에 배운 인증 방식입니다.
3. **본문** — `model` 에는 모델 이름을, `messages` 에는 질문을 넣습니다. 질문은 `role` 이 `"user"` 인 딕셔너리 하나입니다.
4. 무료 등급은 **1분에 15번**까지 호출할 수 있습니다. 넘으면 상태코드 `429` 가 오고, 1분쯤 기다리면 다시 됩니다.

#### 🐍 문법 상자 · `requests.post` 에 넣는 네 가지

```python
response = requests.post(url, headers=headers, json=body, timeout=30)
print(response.status_code)          # 200 — 성공
```

| 쓰는 것 | 뜻 |
|---|---|
| `url` | 보낼 주소 — 맨 앞에 이름 없이 쓴다 |
| `headers=headers` | 요청 헤더 — 키를 여기에 넣는다 |
| `json=body` | 본문 딕셔너리를 JSON 으로 바꿔 보낸다 (9/30) |
| `timeout=30` | 30초 안에 답이 없으면 기다리지 않는다 |

⚠ `requests.post(url, body)` 처럼 이름 없이 쓰면 `body` 가 `json=` 이 아닌 다른 자리로 들어가 요청이 실패합니다. `json=` 을 꼭 붙입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-2 · LLM 을 처음 호출하기</font></h3></td></tr></table>

미리 채워 둔 헤더와 본문으로 LLM 에 요청을 보내고, 상태코드를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `200` |

**💡 힌트**

1. 10/2 오후에 쓴 `requests.post` 와 같은 모양입니다.
2. 괄호 안에 주소, `headers=`, `json=`, `timeout=30` 을 넣습니다.
3. `200` 이 아니면 아침 과제에서 만든 `.env` 의 키를 다시 확인합니다.

In [5]:
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None                                                # 키를 못 찾으면 None 으로 남는다
with open(ENV_PATH, encoding="utf-8") as f:                   # .env 파일을 읽기로 연다
    for line in f:                                            # 파일을 한 줄씩 꺼낸다
        parts = line.strip().split("=", 1)                    # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
        if parts[0] == "GEMINI_API_KEY":                      # = 앞부분이 키 이름이면
            api_key = parts[1]                                # = 뒷부분이 키 값이다
# Gemini 에 요청을 보낼 주소
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
# 요청 헤더 — 키(Bearer 뒤 공백 한 칸)와 본문 형식(JSON)
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# 요청 본문 — 모델 이름과 질문 하나
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
# ── 여기까지 ──

# 1. response 라는 변수를 만들어 url 로 보낸 POST 요청의 결과를 담으세요
#    괄호 안에 url, headers=headers, json=body, timeout=30 을 넣습니다
response = requests.post(url, headers=headers, json=body, timeout=30)

print(response.status_code)

200


#### 🐍 문법 상자 · 안으로 한 단계씩 꺼내기

```python
data = {"choices": [{"message": {"content": "포트 스캔입니다."}}]}
print(data["choices"][0]["message"]["content"])
# 포트 스캔입니다.
```

| 쓰는 것 | 꺼낸 것 | 모양 |
|---|---|---|
| `data["choices"]` | 답 목록 | 리스트 |
| `data["choices"][0]` | 첫 번째 답 | 딕셔너리 |
| `…[0]["message"]` | 메시지 | 딕셔너리 |
| `…["message"]["content"]` | 답 문장 | 문자열 |

⚠ `[0]` 을 빠뜨리면 리스트에 글자 키를 쓴 것이라 `TypeError: list indices must be integers` 가 납니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-3 · 응답에서 답 문장만 꺼내기</font></h3></td></tr></table>

LLM 의 응답에서 **답 문장만** 꺼내 출력하시오. 마지막 줄은 이 요청에 쓴 토큰 합계를 함께 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `198.51.100.7 아이피에서 svc_batch 계정으로 로그인을 시도했으나 실패했습니다.` 와 비슷한 한 문장, 그리고 `쓴 토큰: 85` 같은 숫자 한 줄. 문장과 숫자는 실행할 때마다 조금 달라집니다 |

**💡 힌트**

1. 응답을 딕셔너리로 바꾸는 방법은 9/30 에 배운 `.json()` 입니다.
2. 답 문장은 바깥에서 안으로 한 단계씩 들어가 꺼냅니다 — `choices`, 0번, `message`, `content` 순서입니다.
3. 토큰 수는 `usage` 안의 `total_tokens` 에 있습니다. 마지막 줄에 미리 채워져 있습니다.

In [8]:
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None                                                # 키를 못 찾으면 None 으로 남는다
with open(ENV_PATH, encoding="utf-8") as f:                   # .env 파일을 읽기로 연다
    for line in f:                                            # 파일을 한 줄씩 꺼낸다
        parts = line.strip().split("=", 1)                    # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
        if parts[0] == "GEMINI_API_KEY":                      # = 앞부분이 키 이름이면
            api_key = parts[1]                                # = 뒷부분이 키 값이다
# Gemini 에 요청을 보낼 주소
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
# 요청 헤더 — 키(Bearer 뒤 공백 한 칸)와 본문 형식(JSON)
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# 요청 본문 — 모델 이름과 질문 하나
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
# POST 요청을 보낸다. 30초 안에 답이 없으면 멈춘다
response = requests.post(url, headers=headers, json=body, timeout=30)
# ── 여기까지 ──

# 1. data 라는 변수를 만들어 응답을 딕셔너리로 바꾼 것을 담으세요
data = response.json()
# 2. answer 라는 변수를 만들어 data 에서 꺼낸 답 문장을 담으세요
answer = data["choices"][0]["message"]["content"]

print(answer)
print("쓴 토큰:", data["usage"]["total_tokens"])

2026년 10월 6일 오전 2시 41분 17초에 `198.51.100.7` IP에서 `svc_batch` 계정으로 로그인을 시도했으나 실패했습니다.
쓴 토큰: 118


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.4 되풀이되는 코드를 함수로 묶습니다 — post_to_llm</mark>

문제 2-2 와 2-3 의 셀에는 **미리 채워 둔 줄이 열 줄**쯤 있었습니다. 키를 읽고, 주소와 헤더를 만들고, 요청을 보내는 줄입니다. LLM 을 호출할 때마다 똑같이 되풀이됩니다.

되풀이되는 코드는 **함수 하나로 묶습니다.** 9/23 오후에 배운 방법입니다. 아래 셀의 함수 `post_to_llm` 이 그 일을 합니다.

| 함수 안의 줄 | 하는 일 | 앞에서 직접 쓴 곳 |
|---|---|---|
| `read_api_key()` | `.env` 파일(`ENV_PATH`)에서 API 키를 읽는다 | 문제 2-2 의 미리 채워 둔 여섯 줄 |
| `headers = {"Authorization": "Bearer " + api_key, …}` | 요청 헤더에 키를 넣는다 | 문제 2-2 의 미리 채워 둔 줄 |
| `requests.post(URL, headers=headers, json=body, timeout=30)` | 요청을 보낸다 | 문제 2-2 |
| `return` | 받은 `response` 를 돌려준다 | — |

**쓰는 법은 한 줄입니다.** 본문 `body` 만 만들어서 넣으면 됩니다.

```python
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": "피싱이란 무엇입니까?"}]}

response = post_to_llm(body)
print(response.status_code)          # 200
```

1. 함수가 대신 해 주는 것은 **키 읽기 · 헤더 만들기 · 요청 보내기**입니다. **본문 `body` 는 우리가 만듭니다.** 무엇을 물을지는 본문에 들어 있기 때문입니다.
2. 돌려받은 `response` 는 문제 2-2 의 `response` 와 같은 것입니다. `.status_code` 와 `.json()` 을 그대로 씁니다.
3. 이 아래 문제부터는 셀에 이 함수만 나옵니다. 셀이 짧아지고, **그 문제에서 새로 배우는 줄**만 남습니다.
4. ⚠ 아래 셀을 실행하지 않으면 뒤의 문제에서 `NameError: name 'post_to_llm' is not defined` 가 나옵니다. 노트북을 다시 열었을 때도 아래 셀을 다시 실행합니다.

아래 셀을 실행합니다. 함수를 **만들어 두기만** 합니다. LLM 을 호출하지 않습니다.

In [9]:
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# Gemini 에 요청을 보낼 주소
URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"


def read_api_key():                                           # .env 에서 키를 읽어 돌려주는 함수
    """.env 파일(ENV_PATH)에서 GEMINI_API_KEY 의 값을 읽어 반환합니다."""
    api_key = None                                            # 키를 못 찾으면 None 으로 남는다
    with open(ENV_PATH, encoding="utf-8") as f:               # .env 파일을 읽기로 연다
        for line in f:                                        # 파일을 한 줄씩 꺼낸다
            parts = line.strip().split("=", 1)                # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
            if parts[0] == "GEMINI_API_KEY":                  # = 앞부분이 키 이름이면
                api_key = parts[1]                            # = 뒷부분이 키 값이다
    return api_key                                            # 읽은 키를 돌려준다


def post_to_llm(body, api_key=None):                          # 본문을 받아 Gemini 로 보내는 함수. 키를 주지 않으면 .env 의 키를 쓴다
    """body 를 Gemini API 로 보내고 response 를 반환합니다."""
    if api_key is None:                                       # 키를 따로 주지 않았으면
        api_key = read_api_key()                              # .env 의 키를 쓴다
    # 요청 헤더 — 키(Bearer 뒤 공백 한 칸)와 본문 형식(JSON)
    headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
    # POST 요청을 보내고 응답을 그대로 돌려준다
    response = requests.post(URL, headers=headers, json=body, timeout=30)
    if response.status_code == 429:                           # 한도(1분에 15번)를 넘었으면
        print("[한도 초과] 429 — 1분 기다린 뒤 이 셀을 다시 실행합니다")
    return response                                           # 응답을 그대로 돌려준다


print("post_to_llm 함수를 만들었습니다.")

post_to_llm 함수를 만들었습니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.3 LLM 은 조회하지 않고 문장을 만듭니다</mark>

LLM 은 어딘가를 찾아보고 답하지 않습니다. 그럴듯한 단어를 이어 붙여 문장을 만듭니다. 그래서 **모르는 것도 아는 것처럼 답합니다.** 이것을 환각이라고 합니다.

같은 질문 「IP 211.45.12.9 는 어느 통신사의 주소입니까?」 에 실제로 받은 답입니다.

| 누구에게 물었나 | 받은 답 |
|---|---|
| LLM · 첫 번째 | `LG Uplus` |
| LLM · 두 번째 | `SK텔레콤` |
| 조회 API `ipwho.is` (9/30 에 쓴 서비스) | `SamsungSDS Inc.` |

1. LLM 의 답은 **물을 때마다 달라졌고, 둘 다 틀렸습니다.** 문장은 자연스러워서 틀린 줄 알기 어렵습니다.
2. 조회 API 는 실제 등록 정보를 찾아서 반환합니다.
3. 그래서 **사실 확인은 조회 API 와 로그로** 하고, LLM 에게는 요약과 설명을 맡깁니다. 최종 판단은 사람이 합니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-4 · LLM 의 답과 조회 API 의 답 비교하기</font></h3></td></tr></table>

IP `211.45.12.9` 의 통신사를 LLM 과 조회 API 양쪽에 묻습니다. LLM 의 답 문장과, 조회 API 응답 `truth` 에 든 **통신사 이름**을 나란히 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `LLM     : SK텔레콤` (사람마다 다릅니다) 그리고 `조회 API : SamsungSDS Inc.` |

**💡 힌트**

1. 요청은 `post_to_llm(body)` 로 이미 보냈습니다. 응답을 딕셔너리로 바꾸는 것은 문제 2-3 과 같습니다.
2. `truth` 는 딕셔너리 안에 딕셔너리가 든 모양입니다. 통신사 이름은 `connection` 안의 `isp` 키에 있습니다.
3. 두 줄이 다르면 LLM 이 지어낸 것입니다. 조회 API 쪽이 실제 값입니다.

In [11]:
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
ip = "211.45.12.9"
question = "IP " + ip + " 는 어느 통신사의 주소입니까? 통신사 이름만 답하세요."
# 요청 본문 — 모델 이름과 질문 하나
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
# 조회 API 에 물어 실제 값을 받는다
truth = requests.get("https://ipwho.is/" + ip, timeout=10).json()
response = post_to_llm(body)                                  # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)
# ── 여기까지 ──

# 1. data 라는 변수를 만들어 응답을 딕셔너리로 바꾼 것을 담으세요


answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content

# 2. isp 라는 변수를 만들어 truth 에서 꺼낸 통신사 이름을 담으세요
isp = truth["connection"]["isp"]

print("LLM     :", answer)
print("조회 API :", isp)

LLM     : 2026년 10월 6일 오전 2시 41분 17초에 `198.51.100.7` IP에서 `svc_batch` 계정으로 로그인을 시도했으나 실패했습니다.
조회 API : SamsungSDS Inc.


#### 🐍 문법 상자 · `=` 는 담기, `==` 는 같은지 묻기

```python
status = 400
if status == 200:
    print("성공")
else:
    print("실패:", status)
# 실패: 400
```

| 쓰는 것 | 뜻 |
|---|---|
| `status = 400` | 변수에 값을 담는다 |
| `status == 200` | 같은지 묻는다 — `True` 또는 `False` |
| `else:` | 위 조건이 거짓일 때 |

⚠ `if status = 200:` 처럼 `=` 하나를 쓰면 `SyntaxError` 입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-5 · 키가 틀렸을 때를 대비하기</font></h3></td></tr></table>

일부러 **틀린 키**로 호출했습니다. 요청이 성공했으면 답 문장을, 실패했으면 `실패:` 와 상태코드를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `실패: 400` |

**💡 힌트**

1. 성공인지 확인하는 기준은 상태코드입니다. 성공일 때의 숫자는 9/29 오후에 배웠습니다.
2. 성공했을 때 할 일은 문제 2-3 의 두 줄과 같습니다.
3. 실패한 응답에는 `choices` 키가 없습니다. 확인 없이 꺼내면 프로그램이 멈춥니다.

In [12]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# 요청 본문 — 모델 이름과 질문 하나
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
response = post_to_llm(body, api_key="wrong-key-1234")      # 일부러 틀린 키로 보냅니다
# ── 여기까지 ──

# 1. response 의 상태코드가 200 이면
#    data 라는 변수에 응답을 딕셔너리로 바꿔 담고, 답 문장을 출력하세요
if response.status_code == 200:                               # 200 이면 성공
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    print(data["choices"][0]["message"]["content"])
# 2. 200 이 아니면 "실패:" 와 상태코드를 출력하세요
else:
    print("실패:", response.status_code)

실패: 400


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-6 · 로그 세 줄을 차례로 설명시키기</font></h3></td></tr></table>

`logs` 의 로그 세 줄을 **한 줄씩** LLM 에 보내고, 답 문장을 한 줄씩 출력하시오. 새 문법은 없습니다. 문제 2-3 과 2-5 를 반복문 안에 넣습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 문장 세 줄. 첫 줄은 `jdoe` 의 로그인 성공, 둘째 줄은 `guest01` 의 권한 거부, 셋째 줄은 `deploy` 의 토큰 만료를 설명합니다 |

**💡 힌트**

1. 본문은 로그마다 달라지므로 반복 안에서 만듭니다.
2. 질문 문자열은 `"한 문장으로 설명하세요. 로그: " + log` 로 만듭니다.
3. 답을 꺼내기 전에 상태코드를 확인하는 것은 문제 2-5 와 같습니다. 한 번 실행에 LLM 을 세 번 호출합니다.

In [13]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
logs = [
    "2026-10-06 09:14:52 INFO accepted login for jdoe from 192.0.2.44",
    "2026-10-06 13:07:30 WARN permission denied for guest01 on /etc/shadow",
    "2026-10-06 23:58:11 WARN token expired for deploy from 203.0.113.61",
]
# ── 여기까지 ──

# 1. logs 를 하나씩 꺼내 반복하세요. 꺼낸 로그의 변수 이름은 log 입니다

# 2. 반복 안에서 body 를 만들고, post_to_llm(body) 의 결과를 response 에 담으세요
#    질문은 "한 문장으로 설명하세요. 로그: " + log 입니다

# 3. 상태코드가 200 이면 답 문장을 출력하세요

# 4. 200 이 아니면 "실패:" 와 상태코드를 출력하세요
for log in logs:
    body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": "한 문장으로 설명하세요. 로그: " + log}]}
    response = post_to_llm(body)                              # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)
    if response.status_code == 200:                           # 200 이면 성공
        data = response.json()                                # 응답(JSON 문자열)을 딕셔너리로 바꾼다
        print(data["choices"][0]["message"]["content"])
    else:
        print("실패:", response.status_code)


2026년 10월 6일 오전 9시 14분 52초에 사용자 'jdoe'가 IP 주소 '192.0.2.44'에서 성공적으로 로그인했습니다.
2026년 10월 6일 13시 07분 30초에 게스트 계정(guest01)이 시스템의 중요 보안 파일인 `/etc/shadow`에 접근하려다 권한 부족으로 차단(WARN)되었습니다.
2026년 10월 6일 23시 58분 11초에 IP 주소 203.0.113.61에서 요청된 배포(deploy) 작업의 인증 토큰이 만료되어 경고(WARN)가 발생했습니다.


### 📋 2교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| LLM 에 요청을 보낸다 | `requests.post(url, headers=headers, json=body, timeout=30)` |
| 키를 싣는다 | `{"Authorization": "Bearer " + api_key}` |
| 위 두 줄을 함수로 줄여 쓴다 | `response = post_to_llm(body)` |
| 질문을 담는다 | `"messages": [{"role": "user", "content": question}]` |
| 답 문장을 꺼낸다 | `data["choices"][0]["message"]["content"]` |
| 쓴 토큰을 본다 | `data["usage"]["total_tokens"]` |
| 성공했는지 확인한다 | `response.status_code == 200` |

LLM 의 답은 문장이 자연스러워도 틀릴 수 있습니다. 사실 확인은 조회 API 와 로그로 합니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">3교시 (11:00–11:50) · 프롬프트로 답의 모양을 정한다</mark>

3교시 문제는 2교시의 `post_to_llm` 함수를 씁니다. `NameError: name 'post_to_llm' is not defined` 가 나오면 **`2.4` 의 함수 셀을 다시 실행**합니다.

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 상관없습니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **프롬프트 (prompt)** | LLM 에게 보내는 글에 무엇을 적어야 답이 좋아지는가 |
| **temperature** | 이 숫자는 무엇을 정하고, 자동화에서는 왜 낮게 두는가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 프롬프트 →
- temperature →

적어 둔 것은 3교시가 끝날 때 다시 봅니다.

---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3 · 프롬프트로 답의 모양을 정한다</mark>

### 왜 필요한가

1. 로그 한 줄만 보내면 LLM 은 **영어로, 항목을 여러 개 나열해** 길게 답했습니다. 실제로 229토큰이 나왔습니다.
2. 답의 길이와 모양이 매번 다르면, 뒤에 오는 파이썬 코드가 값을 꺼낼 수 없습니다.
3. LLM 에게 보내는 글 전체를 **프롬프트**라고 합니다. 프롬프트에 **역할 · 지시 · 출력 형식 · 예시**를 적으면 답의 모양이 정해집니다.

### 3교시에 사용하는 용어

| 용어 (읽는 법) | 뜻 | 3교시의 예 |
|---|---|---|
| 프롬프트 (prompt) | 모델에게 건네는 지시문 전체 | `messages` 에 담는 글 |
| system · user (시스템 · 유저) | 역할 지시와 이번 질문을 나눠 담는 자리 | `{"role": "system", ...}` · `{"role": "user", ...}` |
| few-shot (퓨샷) | 본보기를 예시로 주는 방식 | `예시: {"severity": "high", "summary": "root 로그인 실패"}` |
| temperature (템퍼러처) | 답을 얼마나 다양하게 낼지 정하는 값 | `"temperature": 0` |

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.1 역할은 system 에, 질문은 user 에 적습니다</mark>

`messages` 리스트에 딕셔너리를 **둘** 넣습니다. 앞의 것이 역할, 뒤의 것이 이번 질문입니다.

```python
messages = [
    {"role": "system", "content": "당신은 보안관제 애널리스트입니다. 한 문장으로만 답합니다."},
    {"role": "user", "content": "2026-10-06 03:22:40 WARN failed login for ops.kim from 203.0.113.61"}
]
body = {"model": "gemini-3.5-flash-lite", "messages": messages}
response = post_to_llm(body)
```

같은 로그를 보냈을 때 실제로 받은 답입니다.

| 보낸 것 | 받은 답 | 토큰 |
|---|---|---|
| 로그만 | 영어로 항목 여러 개를 나열한 긴 설명 | 271 |
| 역할 + 로그 | 「203.0.113.61 IP에서 ops.kim 계정으로의 로그인 실패(WARN)를 나타내며 … 모니터링이 필요합니다.」 | 132 |

1. 역할을 정해 주면 쓰는 단어와 판단 기준이 보안 직무 쪽으로 좁혀집니다.
2. 「한 문장으로만」 같은 규칙도 `system` 에 적습니다.
3. 답이 짧아지면 토큰도 줄어듭니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
messages = [
    {"role": "system", "content": "당신은 보안관제 애널리스트입니다."},
    {"role": "user", "content": "이 로그가 위험합니까?"}
]

print(messages[0]["role"])
```

막히면 바로 위 `3.1 역할은 system 에, 질문은 user 에 적습니다` 설명을 다시 봅니다.

In [14]:
messages = [
    {"role": "system", "content": "당신은 보안관제 애널리스트입니다."},
    {"role": "user", "content": "이 로그가 위험합니까?"}
]

print(messages[0]["role"])

system


✅ `system`

#### 🐍 문법 상자 · 딕셔너리 두 개가 든 리스트

```python
messages = [
    {"role": "system", "content": "당신은 보안관제 애널리스트입니다."},
    {"role": "user", "content": "이 로그가 위험합니까?"},
]
print(len(messages), messages[1]["role"])
# 2 user
```

| 쓰는 것 | 결과 |
|---|---|
| `messages[0]` | 첫 딕셔너리 — 역할(`system`) |
| `messages[1]` | 둘째 딕셔너리 — 질문(`user`) |
| `messages[1]["role"]` | `"user"` |

⚠ 딕셔너리와 딕셔너리 사이에 쉼표가 없으면 `SyntaxError` 입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-2 · 역할을 넣어 호출하기</font></h3></td></tr></table>

역할을 `system` 에, 로그를 `user` 에 담아 LLM 을 호출하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `외부 IP(198.51.100.23)에서 root 계정을 대상으로 발생한 무차별 대입 공격 시도로 판단됩니다.` 와 비슷한 **한국어 한 문장** |

**💡 힌트**

1. `messages` 는 딕셔너리 둘이 든 리스트입니다. 첫째의 `role` 은 `system`, 둘째는 `user` 입니다.
2. `system` 의 `content` 에는 역할과 「한 문장으로만 답합니다」를 적습니다. `user` 의 `content` 에는 변수 `log` 를 넣습니다.
3. 본문의 `messages` 값에 변수 `messages` 를 넣습니다. 요청은 미리 채워 둔 `post_to_llm(body)` 가 보냅니다.

In [15]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
log = "2026-10-06 01:48:09 WARN failed login for root from 198.51.100.23"
# ── 여기까지 ──

# 1. messages 라는 변수를 만들어 system 딕셔너리와 user 딕셔너리를 리스트에 담으세요
#    system 의 content 는 "당신은 보안관제 애널리스트입니다. 한 문장으로만 답합니다." 입니다


# 2. body 라는 변수를 만들어 모델 이름과 messages 를 넣은 본문을 담으세요


response = post_to_llm(body)                                  # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)

data = response.json()                                        # 응답(JSON 문자열)을 딕셔너리로 바꾼다
answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content
print(answer)

2026년 10월 6일 23시 58분 11초에 IP 주소 203.0.113.61에서 요청된 배포(deploy) 작업의 인증 토큰이 만료되었다는 경고(WARN) 로그입니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.2 지시와 출력 형식을 분명하게 적습니다</mark>

무엇을 할지(**지시**)와 어떤 모양으로 답할지(**출력 형식**)를 질문에 적습니다. 같은 로그 `ops.kim` 에 실제로 받은 답입니다.

| 질문에 적은 것 | 받은 답 |
|---|---|
| 「심각도를 high, medium, low 중 하나로만 답하세요.」 | `low` |
| 「JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다.」 | 아래 |

```
```json
{
  "severity": "medium",
  "summary": "IP 주소 203.0.113.61에서 계정 ops.kim으로 로그인 실패 발생"
}
```
```

1. 고를 수 있는 값을 **나열해 주면** 그 안에서만 답합니다.
2. JSON 으로 받으면 뒤에 오는 코드가 `severity` 같은 **키 이름으로 값을 꺼낼 수 있습니다.**
3. 받은 글의 앞뒤에 `` ```json `` 과 `` ``` `` 이 붙어 있습니다. 4교시에 이 표시를 지우고 읽습니다.
4. 같은 로그인데 한 단어로 물으면 `low`, JSON 으로 물으면 `medium` 이 나왔습니다. **LLM 의 판단은 묻는 방식에 따라 달라집니다.**

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-3 · 답을 JSON 으로 받기</font></h3></td></tr></table>

심각도와 한 줄 요약을 **JSON 으로** 답하게 하시오. 키는 `severity` 와 `summary` 입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 코드 블록 표시(```json)가 붙은 JSON 문자열. `severity` 는 `high` 입니다. `summary` 는 길게 나오거나 영어로 나올 수 있습니다 |

**💡 힌트**

1. 질문 문자열에 「JSON 으로만 답하세요」와 키 이름 둘을 적습니다.
2. `severity` 에 들어갈 수 있는 세 단어도 질문에 적습니다.
3. 받은 글은 아직 **문자열**입니다. 딕셔너리로 바꾸는 것은 4교시에 합니다.

In [16]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
log = "2026-10-06 01:48:09 WARN failed login for root from 198.51.100.23"
# ── 여기까지 ──

# 1. question 이라는 변수를 만들어 아래 지시 뒤에 log 를 이어 붙이세요
#    "다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다. 로그: "
question = "다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다. 로그: " + log

# 2. messages 라는 변수를 만들어 system 딕셔너리와 user 딕셔너리를 담으세요
messages = [{"role": "system", "content": "당신은 보안관제 애널리스트입니다."}, {"role": "user", "content": question}]

# 요청 본문 — 모델 이름과 messages
body = {"model": "gemini-3.5-flash-lite", "messages": messages}

response = post_to_llm(body)                                  # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)

data = response.json()                                        # 응답(JSON 문자열)을 딕셔너리로 바꾼다
answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content
print(answer)

```json
{
  "severity": "high",
  "summary": "198.51.100.23 IP에서 root 계정으로의 로그인 실패(브루트 포스 시도 가능성)"
}
```


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.3 예시를 보여 주면 그 모양을 따라 합니다</mark>

본보기를 한두 개 보여 주는 방식을 **few-shot** 이라고 합니다. 질문에 예시 한 줄을 넣었을 때 실제로 달라진 것입니다.

| 질문 | 받은 `summary` |
|---|---|
| 예시 없음 | `IP 주소 203.0.113.61에서 사용자 계정 ops.kim으로의 로그인 시도가 실패했습니다.` |
| `예시: {"severity": "high", "summary": "root 로그인 실패"}` 를 넣음 | `ops.kim 계정 로그인 실패` |

1. 예시가 짧으면 답도 짧아집니다. 길이를 글로 설명하는 것보다 **예시 하나가 더 잘 통합니다.**
2. 예시의 **키 이름과 순서, 쓰는 언어**도 그대로 따라 합니다. 예시가 없으면 요약이 영어로 올 때가 있습니다.
3. 예시를 넣어도 `` ```json `` 표시는 붙을 때도 있고 안 붙을 때도 있었습니다. 세 번 가운데 두 번 붙었습니다.

#### 🐍 문법 상자 · 문자열을 이어 붙일 때는 공백도 직접 넣습니다

```python
a = "JSON 으로만 답하세요."
b = "예시: high"
print(a + b)
print(a + " " + b)
# JSON 으로만 답하세요.예시: high
# JSON 으로만 답하세요. 예시: high
```

| 쓰는 것 | 결과 |
|---|---|
| `a + b` | 두 문장이 붙어 버린다 |
| `a + " " + b` | 사이에 공백 한 칸 |

⚠ `+` 는 공백을 넣어 주지 않습니다. 붙어 버린 질문은 LLM 도 읽기 어렵습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-4 · 예시를 넣어 요약을 짧게 받기</font></h3></td></tr></table>

문제 3-3 의 질문에 **예시 한 줄**을 더해, `summary` 가 예시처럼 짧게 나오게 하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `summary` 가 `root 계정 로그인 실패` 처럼 예시와 같은 길이의 **한국어**로 나옵니다 |

**💡 힌트**

1. 질문 문자열을 `지시 + 예시 + 로그` 순서로 이어 붙입니다.
2. 예시는 `example` 에 미리 들어 있습니다. 문자열 사이에는 공백이 한 칸씩 있어야 읽기 쉽습니다.
3. 예시가 없을 때의 답(문제 3-3)과 `summary` 길이를 견줘 봅니다.

In [17]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
log = "2026-10-06 01:48:09 WARN failed login for root from 198.51.100.23"
instruction = "다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다."
example = '예시: {"severity": "high", "summary": "admin 로그인 실패"}'
# ── 여기까지 ──

# 1. question 이라는 변수를 만들어 instruction, example, "로그: " + log 를 공백 한 칸씩 두고 이어 붙이세요
question = instruction + " " + example + " 로그: " + log

# 2. messages 라는 변수를 만들어 system 딕셔너리와 user 딕셔너리를 담으세요
messages = [{"role": "system", "content": "당신은 보안관제 애널리스트입니다."}, {"role": "user", "content": question}]

# 요청 본문 — 모델 이름과 messages
body = {"model": "gemini-3.5-flash-lite", "messages": messages}

response = post_to_llm(body)                                  # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)

data = response.json()                                        # 응답(JSON 문자열)을 딕셔너리로 바꾼다
answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content
print(answer)

```json
{
  "severity": "high",
  "summary": "root 계정 로그인 실패 (IP: 198.51.100.23)"
}
```


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.4 temperature 를 0 으로 두어도 답은 달라질 수 있습니다</mark>

본문에 `"temperature"` 를 넣으면 답을 얼마나 다양하게 낼지를 정할 수 있습니다. 보안 자동화에서는 **낮게(0)** 둡니다.

```python
body = {"model": "gemini-3.5-flash-lite", "temperature": 0, "messages": messages}
```

그런데 0 으로 두어도 답이 똑같지는 않았습니다. 「로그인 실패가 다섯 번 이어진 사용자에게 보낼 경고 문구를 한 줄로 쓰세요.」 를 **temperature 0** 으로 세 번 물었을 때 실제로 받은 답입니다.

| 몇 번째 | 받은 답 |
|---|---|
| 1 | 비밀번호 오류 5회 초과로 계정이 임시 잠겼습니다. 본인 확인 후 잠금을 해제해 주세요. |
| 2 | 로그인 5회 실패로 보안을 위해 계정이 잠겼으니 비밀번호를 재설정해 주세요. |
| 3 | 비밀번호를 5회 연속 잘못 입력하여 계정이 일시 잠금되었습니다. |

같은 temperature 0 으로 「심각도를 high, medium, low 중 하나로만 답하세요.」 라고 세 번씩 물었을 때입니다.

| 로그 | 받은 답 세 개 |
|---|---|
| `ops.kim` 로그인 실패 | `medium` · `medium` · `medium` |
| `root` 로그인 실패 | `high` · `medium` · `high` |

1. temperature 를 0 으로 두어도 **답은 매번 달라질 수 있습니다.** 문장도, 심각도 판단도 그렇습니다.
2. 그래도 **출력 형식을 좁히면** 답이 세 단어 가운데 하나로 모입니다. 문장은 비교할 수 없지만 단어는 코드로 세고 비교할 수 있습니다.
3. 같은 로그에 `high` 와 `medium` 이 섞여 나왔습니다. 그래서 **LLM 의 판단을 그대로 믿지 않고 사람이 최종 확인**합니다.
4. temperature 를 0 으로 해도 **환각은 사라지지 않습니다.** 2교시의 통신사 질문은 0 으로 물어도 `KT`, `LG Uplus` 로 틀린 답이 나왔습니다.

#### 🐍 문법 상자 · `temperature` 는 본문 딕셔너리의 키입니다

```python
body = {"model": "gemini-3.5-flash-lite", "temperature": 0, "messages": []}
print(list(body.keys()))
# ['model', 'temperature', 'messages']
```

| 키 | 넣는 것 |
|---|---|
| `model` | 모델 이름 |
| `temperature` | `0` — 답을 매번 비슷하게 |
| `messages` | 역할 · 질문 리스트 |

⚠ `temperature` 를 `messages` 안의 딕셔너리에 넣지 않습니다. `model` · `messages` 와 같은 자리입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-5 · 네 조각을 한 프롬프트에 모으기</font></h3></td></tr></table>

역할 · 지시 · 출력 형식 · 예시를 모두 넣고 **temperature 0** 으로 호출하시오. 새 로그는 `svc_backup` 의 권한 거부입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `severity` 가 `high` 이고 `summary` 가 `svc_backup` 의 권한 거부를 한 줄로 요약한 JSON 문자열 |

**💡 힌트**

1. 질문 문자열은 문제 3-4 와 같은 순서로 이어 붙입니다 — 지시, 예시, 로그.
2. 역할은 `messages` 의 `system` 에 들어갑니다.
3. `temperature` 는 `messages` 가 아니라 본문 `body` 에 넣습니다. `model` · `messages` 와 같은 자리입니다.

In [18]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
log = "2026-10-06 14:31:27 WARN permission denied for svc_backup on /etc/sudoers"
instruction = "다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다."
example = '예시: {"severity": "high", "summary": "admin 로그인 실패"}'
# ── 여기까지 ──

# 1. question 이라는 변수를 만들어 instruction, example, "로그: " + log 를 이어 붙이세요
question = instruction + " " + example + " 로그: " + log

# 2. messages 라는 변수를 만들어 system 딕셔너리와 user 딕셔너리를 담으세요. system 의 content 는 "당신은 보안관제 애널리스트입니다." 입니다
messages = [{"role": "system", "content": "당신은 보안관제 애널리스트입니다."}, {"role": "user", "content": question}]


# 3. body 라는 변수를 만들어 모델 이름, temperature 0, messages 를 넣은 본문을 담으세요
body = {"model": "gemini-3.5-flash-lite", "temperature": 0, "messages": messages}

response = post_to_llm(body)                                  # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)

data = response.json()                                        # 응답(JSON 문자열)을 딕셔너리로 바꾼다
answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content
print(answer)

```json
{
  "severity": "high",
  "summary": "svc_backup 계정의 /etc/sudoers 파일 접근 권한 거부 발생"
}
```


#### 🐍 문법 상자 · 반복하며 하나씩 세기

```python
number = 0
for log in ["a", "b", "c"]:
    number = number + 1
    print(number, log)
# 1 a
# 2 b
# 3 c
```

| 줄 | 하는 일 |
|---|---|
| `number = 0` | 반복 **전에** 한 번 0 으로 둔다 |
| `number = number + 1` | 반복할 때마다 1 씩 늘린다 |

⚠ `number = 0` 을 반복 안에 두면 매번 0 으로 돌아가 모두 `1` 이 나옵니다.

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-6 · 로그 세 줄의 심각도를 한 줄씩 받기</font></h3></td></tr></table>

`logs` 의 로그 세 줄을 한 줄씩 보내 **심각도 한 단어**를 받고, `로그 번호 → 심각도` 형식으로 출력하시오. temperature 는 0 입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `1 → low` · `2 → High` · `3 → high` 처럼 세 줄. 단어와 대소문자는 달라질 수 있습니다 |

**💡 힌트**

1. 반복 안에서 `messages`, `body`, 요청, 답 꺼내기를 차례로 합니다. 문제 3-2 와 같은 방법입니다.
2. 몇 번째 로그인지는 반복 전에 `number = 0` 을 두고 반복 안에서 1 씩 더해 셉니다.
3. 출력은 `print(number, "→", answer)` 로 합니다. 한 번 실행에 LLM 을 세 번 호출합니다.

In [19]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
logs = [
    "2026-10-06 09:02:11 INFO accepted login for lee.yh from 192.0.2.17",
    "2026-10-06 02:15:44 WARN failed login for root from 198.51.100.23",
    "2026-10-06 02:16:03 INFO accepted login for root from 198.51.100.23",
]
instruction = "다음 로그의 심각도를 high, medium, low 중 하나로만 답하세요. 로그: "
# ── 여기까지 ──

# 1. number 라는 변수를 만들어 0 을 담으세요

# 2. logs 를 하나씩 꺼내 반복하세요. 꺼낸 로그의 변수 이름은 log 입니다

# 3. 반복 안에서 number 에 1 을 더하고, messages 와 body(temperature 0) 를 만들어 post_to_llm(body) 로 보내세요

# 4. 응답에서 답 단어를 꺼내 answer 에 담으세요

# 5. number, "→", answer 를 출력하세요
number = 0                                                    # 몇 번째 로그인지 센다
for log in logs:
    number = number + 1                                       # 하나 늘린다
    messages = [{"role": "system", "content": "당신은 보안관제 애널리스트입니다."}, {"role": "user", "content": instruction + log}]
    # temperature 0 — 답이 덜 흔들린다
    body = {"model": "gemini-3.5-flash-lite", "temperature": 0, "messages": messages}
    response = post_to_llm(body)                              # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    answer = data["choices"][0]["message"]["content"]         # 답 문장 꺼내기: choices → 0번 → message → content
    print(number, "→", answer)

1 → low
2 → medium
3 → High


### 📋 3교시 한눈에

| 프롬프트의 조각 | 어디에 적나 | 무엇이 달라지나 |
|---|---|---|
| 역할 | `messages` 의 `system` | 쓰는 단어와 판단 기준 |
| 지시 | `user` 의 `content` | 무엇을 할지 |
| 출력 형식 | `user` 의 `content` | 답의 모양 (한 단어 · JSON) |
| 예시 | `user` 의 `content` | 답의 길이와 키 이름 |
| temperature | 본문 `body` | 답의 다양함. 자동화에서는 0 으로 둔다 |

temperature 를 0 으로 두어도 답은 달라질 수 있고 환각도 남습니다. 출력 형식을 좁히면 답이 달라져도 코드가 다룰 수 있는 값이 됩니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">4교시 (12:00–12:50) · JSON 답을 안전하게 읽는다 — llm_client.py</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **파싱 (parsing)** | 문자열로 받은 것을 프로그램이 쓸 수 있는 값으로 바꾼다는 것이 무슨 뜻인가 |
| **구조화된 출력 (structured output)** | LLM 의 답을 문장이 아니라 JSON 으로 받으면 무엇이 좋은가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 파싱 →
- 구조화된 출력 →

---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4 · JSON 답을 안전하게 읽는다</mark>

### 왜 필요한가

1. 3교시에 JSON 으로 답하라고 했더니, 답의 앞뒤에 `` ```json `` 과 `` ``` `` 이 붙어서 왔습니다. 세 번 물으면 세 번 다 붙었습니다.
2. 그 문자열을 `json.loads` 에 그대로 넣으면 `JSONDecodeError` 로 프로그램이 멈춥니다.
3. 로그 백 건을 처리하다가 한 건에서 멈추면 남은 아흔아홉 건을 잃습니다. **읽지 못한 한 건은 건너뛰고 계속 가야** 합니다.

### 4교시에 사용하는 용어

| 용어 (읽는 법) | 뜻 | 4교시의 예 |
|---|---|---|
| 파싱 (parsing) | 문자열을 프로그램이 쓰는 값으로 바꾸는 일 | `json.loads(clean)` |
| 코드 블록 표시 | 답의 앞뒤에 붙는 `` ```json `` 과 `` ``` `` | `text.replace("```json", "")` |
| `call_llm` | 질문을 받아 LLM 의 응답 문자열을 반환하는 함수 | `call_llm(question)` |
| `parse_llm_json` | 응답 문자열을 딕셔너리로 바꾸는 함수. 못 읽으면 `None` | `parse_llm_json(text)` |

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 표시를 지우고 나서 읽습니다</mark>

3교시에 실제로 받은 답은 이런 문자열이었습니다.

```
```json
{
  "severity": "medium",
  "summary": "IP 주소 203.0.113.61에서 계정 ops.kim으로 로그인 실패 발생"
}
```
```

이 문자열을 그대로 `json.loads` 에 넣으면 실제로 이런 오류가 납니다.

```
JSONDecodeError: Expecting value: line 1 column 1 (char 0)
```

첫 글자가 `{` 가 아니라 `` ` `` 이기 때문입니다. 그래서 표시를 먼저 지웁니다.

```python
import json

text = '```json{"severity": "medium", "summary": "ops.kim 로그인 실패"}```'

clean = text.replace("```json", "")
clean = clean.replace("```", "")
clean = clean.strip()

result = json.loads(clean)
print(result["summary"])          # ops.kim 로그인 실패
```

1. `replace` 는 9/23 에, `strip` 은 9/28 에, `json.loads` 는 9/29 오후에 배웠습니다. 새 문법은 없습니다.
2. `` ```json `` 을 먼저 지우고 `` ``` `` 을 나중에 지웁니다. 순서를 바꾸면 `json` 문자열이 남습니다.
3. `strip` 은 앞뒤에 남은 공백과 줄바꿈을 지웁니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
text = '```json{"severity": "low"}```'

clean = text.replace("```json", "")
clean = clean.replace("```", "")

print(clean)
```

막히면 바로 위 `4.1 표시를 지우고 나서 읽습니다` 설명을 다시 봅니다.

In [20]:
text = '```json{"severity": "low"}```'

clean = text.replace("```json", "")
clean = clean.replace("```", "")

print(clean)

{"severity": "low"}


✅ `{"severity": "low"}`

#### 🐍 문법 상자 · 지우는 순서가 결과를 바꿉니다

```python
text = '```json{"a": 1}```'
print(text.replace("```", "").replace("```json", ""))
print(text.replace("```json", "").replace("```", ""))
# json{"a": 1}
# {"a": 1}
```

| 순서 | 결과 |
|---|---|
| `"```"` 먼저 | `json` 글자가 남는다 — `json.loads` 가 읽지 못한다 |
| `"```json"` 먼저 | 깨끗한 JSON 만 남는다 |

⚠ 긴 표시를 먼저 지웁니다. 짧은 것을 먼저 지우면 긴 것의 일부만 지워집니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-2 · 표시를 지우고 딕셔너리로 바꾸기</font></h3></td></tr></table>

LLM 의 답 `text` 에서 코드 블록 표시를 지우고 딕셔너리로 바꿔, 심각도와 요약을 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `high` 그리고 `root 계정 로그인 실패 4회` |

**💡 힌트**

1. 표시는 둘입니다. 문자열이 긴 쪽을 먼저 지웁니다.
2. 지운 뒤에는 앞뒤 공백도 지웁니다.
3. 문자열을 딕셔너리로 바꾸는 것은 9/29 오후에 배운 `json.loads` 입니다.

In [24]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
text = '```json {"severity": "high", "summary": "root 계정 로그인 실패 4회"} ```'
# ── 여기까지 ──

# 1. clean 이라는 변수를 만들어 text 에서 "```json" 을 지운 문자열을 담으세요
clean = text.replace("```json", "")
# 2. clean 에서 "```" 을 지운 문자열을 다시 clean 에 담으세요
clean = clean.replace("```", "")

# 3. clean 의 앞뒤 공백을 지운 문자열을 다시 clean 에 담으세요
clean = clean.strip()

# 4. result 라는 변수를 만들어 clean 을 딕셔너리로 바꾼 것을 담으세요
result = json.loads(clean)

print(result["severity"])
print(result["summary"])

high
root 계정 로그인 실패 4회


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.2 읽지 못해도 멈추지 않게 합니다</mark>

LLM 이 JSON 이 아닌 문장으로 답할 때가 있습니다. 그때 `json.loads` 는 `JSONDecodeError` 를 냅니다. 9/28 에 배운 `try` · `except` 로 감쌉니다.

```python
import json

text = "이 로그는 분석할 수 없습니다."

try:
    result = json.loads(text)
except json.JSONDecodeError:
    result = None

print(result)          # None
```

1. 읽는 데 성공하면 `result` 에 딕셔너리가 담깁니다.
2. 실패하면 `result` 에 `None` 이 담기고, 프로그램은 다음 줄로 넘어갑니다.
3. 부르는 쪽은 `if result:` 로 확인합니다. 9/30 의 `call_with_retry` 와 같은 약속입니다.

#### 🐍 문법 상자 · `try` · `except` — 읽지 못해도 멈추지 않게

```python
import json
for s in ['{"a": 1}', "그럴듯한 문장"]:
    try:
        print(json.loads(s))
    except json.JSONDecodeError:
        print("읽지 못함")
# {'a': 1}
# 읽지 못함
```

| 들어온 글 | 실행되는 곳 |
|---|---|
| JSON 모양 | `try` 안 — 딕셔너리가 된다 |
| 그냥 문장 | 에러가 나서 `except` 안으로 (9/28) |

⚠ `except` 가 없으면 LLM 이 문장으로 답한 순간 프로그램이 멈춥니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-3 · parse_llm_json 함수 만들기</font></h3></td></tr></table>

응답 문자열을 받아 딕셔너리를 반환하는 함수 `parse_llm_json` 을 완성하시오. 읽지 못하면 `None` 을 반환해 프로그램이 멈추지 않게 합니다. 표시를 지우는 세 줄은 문제 4-2 에서 쓴 것이라 미리 채워 두었습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{'severity': 'low'}` 그리고 `None` |

**💡 힌트**

1. 해 볼 일은 `try:` 아래에, 실패했을 때 할 일은 `except` 아래에 씁니다. 바로 위 `4.2` 의 코드와 같은 모양입니다.
2. 성공하면 `return` 으로 딕셔너리를 반환하고, 실패하면 `return None` 입니다.
3. `try` 와 `except` 아래 줄은 여덟 칸 들여 씁니다.

In [28]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구


def parse_llm_json(text):                                     # LLM 의 답을 딕셔너리로 바꾸는 함수. 못 읽으면 None
    clean = text.replace("```json", "")                       # 코드 블록 표시 ```json 을 지운다
    clean = clean.replace("```", "")                          # 남은 ``` 를 지운다
    clean = clean.strip()                                     # 앞뒤 공백 · 줄바꿈을 지운다

    try:                                                      # 아래 줄을 해 본다
        # 1. clean 을 딕셔너리로 바꾼 것을 return 하세요
        cLean_dict = json.loads(clean)
        return cLean_dict
 
    except json.JSONDecodeError:                              # JSON 으로 읽지 못하면 여기로 온다
        # 2. None 을 return 하세요
        return None



print(parse_llm_json('```json{"severity": "low"}```'))
print(parse_llm_json("분석할 수 없습니다."))

{'severity': 'low'}
None


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.3 호출도 함수로 묶습니다 — call_llm</mark>

`post_to_llm` 은 요청을 보내는 데까지만 합니다. 본문을 만들고, 상태코드를 확인하고, 답 문장을 꺼내는 줄은 3교시 내내 되풀이했습니다. 이 줄들도 함수 하나에 넣습니다. **질문을 주면 응답 문자열을 반환합니다.**

```python
def call_llm(question):
    body = {"model": "gemini-3.5-flash-lite", "temperature": 0,
            "messages": [{"role": "user", "content": question}]}
    response = post_to_llm(body)
    if response.status_code != 200:
        print("[LLM 호출 실패] 상태 코드", response.status_code)      # 429 면 1분 기다린 뒤 다시 실행
        return ""
    data = response.json()
    return data["choices"][0]["message"]["content"]
```

1. 실패하면 **빈 문자열 `""`** 를 반환합니다. 빈 문자열은 `parse_llm_json` 이 읽지 못해서 `None` 이 됩니다. 두 함수가 그렇게 이어집니다.
2. temperature 는 0 으로 고정합니다.
3. 함수로 묶으면 부르는 쪽은 한 줄이면 됩니다 — `call_llm("…")`.

#### 🐍 문법 상자 · `%%writefile` 은 셀 맨 첫 줄에

```python
%%writefile llm_client.py
import json
...
```

| 할 일 | 결과 |
|---|---|
| 셀 첫 줄에 `%%writefile 파일이름` | 셀 내용이 그 파일로 저장된다 |
| 파일을 고친 뒤 | 이 셀을 **다시 실행**해야 파일이 바뀐다 |

⚠ `%%writefile` 위에 빈 줄이나 주석이 있으면 저장되지 않고 에러가 납니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-4 · llm_client.py 만들기</font></h3></td></tr></table>

오전의 산출물 **`llm_client.py`** 를 만드시오. 두 함수를 파일 하나에 담습니다. 새 문법은 없습니다.

파일은 **노트북 없이 혼자 실행돼야** 합니다. 그래서 파일 안에는 `post_to_llm` 대신 키를 읽는 `read_api_key` 와 `requests.post` 가 그대로 들어 있습니다. `.env` 를 찾는 `find_env` 도 파일 안에 미리 들어 있습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `Writing llm_client.py` (다시 실행하면 `Overwriting llm_client.py`) |

**💡 힌트**

1. 첫 줄 `%%writefile llm_client.py` 는 셀의 내용을 파일로 저장합니다. 9/23 오후부터 쓴 방법입니다.
2. 파일 안에서는 주소가 `URL`, 모델 이름이 `MODEL` 이라는 변수에 들어 있습니다.
3. 빈 줄을 채우지 않고 저장하면 다음 문제에서 오류가 납니다. 두 군데를 다 채운 뒤 실행합니다. 고쳤으면 이 셀을 다시 실행해 저장한 뒤 다음 문제 셀을 다시 실행합니다.

In [29]:
%%writefile llm_client.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import os                                                     # 폴더 · 파일을 다루는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# Gemini 에 요청을 보낼 주소
URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL = "gemini-3.5-flash-lite"                               # 쓸 모델 이름


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                          # 지금 폴더에서 다섯 칸 위까지 찾습니다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 돌려준다
        folder = os.path.dirname(folder)        # 한 칸 위 폴더
    return None


def read_api_key():                                           # .env 에서 키를 읽어 돌려주는 함수
    api_key = None                                            # 키를 못 찾으면 None 으로 남는다
    with open(find_env(), encoding="utf-8") as f:             # .env 를 찾아 읽기로 연다
        for line in f:                                        # 파일을 한 줄씩 꺼낸다
            parts = line.strip().split("=", 1)                # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
            if parts[0] == "GEMINI_API_KEY":                  # = 앞부분이 키 이름이면
                api_key = parts[1]                            # = 뒷부분이 키 값이다
    return api_key                                            # 읽은 키를 돌려준다


def call_llm(question):                                       # 질문을 받아 답 문장을 돌려주는 함수
    # 요청 헤더 — .env 에서 읽은 키를 넣는다
    headers = {"Authorization": "Bearer " + read_api_key(), "Content-Type": "application/json"}
    # 1. body 라는 변수를 만들어 MODEL, temperature 0, question 을 넣은 본문을 담으세요
body = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": question}]}

    # 2. response 라는 변수를 만들어 URL 로 보낸 POST 요청의 결과를 담으세요
response = requests.post(URL, hraders=headers, json=body, timeout=30)

    if response.status_code != 200:                           # 200(성공)이 아니면
        print("[LLM 호출 실패] 상태 코드", response.status_code)      # 429 면 1분 기다린 뒤 다시 실행
        return ""                                             # 실패하면 빈 문자열을 돌려준다
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    return data["choices"][0]["message"]["content"]           # 답 문장만 돌려준다


def parse_llm_json(text):                                     # LLM 의 답을 딕셔너리로 바꾸는 함수. 못 읽으면 None
    clean = text.replace("```json", "")                       # 코드 블록 표시 ```json 을 지운다
    clean = clean.replace("```", "")                          # 남은 ``` 를 지운다
    clean = clean.strip()                                     # 앞뒤 공백 · 줄바꿈을 지운다

    try:                                                      # 아래 줄을 해 본다
        return json.loads(clean)                              # 문자열을 딕셔너리로 바꿔 돌려준다
    except json.JSONDecodeError:                              # JSON 으로 읽지 못하면 여기로 온다
        return None

Writing llm_client.py


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.4 만든 파일을 불러 씁니다</mark>

`llm_client.py` 는 같은 폴더에 있으므로 `import` 로 불러 씁니다. 9/23 오후에 배운 모듈 불러오기입니다.

```python
from llm_client import call_llm, parse_llm_json

text = call_llm("JSON 으로만 답하세요. 키는 severity 하나입니다. 로그: …")
result = parse_llm_json(text)

if result:
    print(result["severity"])
else:
    print("읽지 못했습니다")
```

1. `call_llm` 이 응답 문자열을 주고, `parse_llm_json` 이 그 문자열을 딕셔너리로 바꿉니다.
2. `result` 가 `None` 일 수 있으므로 **꺼내기 전에 `if result:` 로 확인합니다.**
3. `llm_client.py` 를 고친 뒤에는 문제 4-4 셀을 다시 실행해 저장하고, 이 문제 셀을 다시 실행합니다.

#### 🐍 문법 상자 · `if result:` — 값이 있을 때만

```python
for result in [{"severity": "high"}, None]:
    if result:
        print("있음", result["severity"])
    else:
        print("없음")
# 있음 high
# 없음
```

| `result` | `if result:` |
|---|---|
| 딕셔너리 | 참 — 키로 꺼낸다 |
| `None` | 거짓 — `else` 로 간다 |

⚠ 확인 없이 `None["severity"]` 를 쓰면 `TypeError: 'NoneType' object is not subscriptable` 입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-5 · 두 함수를 이어 로그 한 줄 분석하기</font></h3></td></tr></table>

`llm_client.py` 의 두 함수로 로그를 분석하고, 심각도와 요약을 한 줄로 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `low | root 계정 로그인 성공` 과 비슷한 한 줄 |

**💡 힌트**

1. `call_llm` 에 `prompt` 를 넣으면 응답 문자열이 반환됩니다.
2. 그 문자열을 `parse_llm_json` 에 넣으면 딕셔너리 또는 `None` 이 반환됩니다.
3. `None` 인지 먼저 확인하고 나서 키로 값을 꺼냅니다.

In [33]:
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)          # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
prompt = '다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다. '
prompt = prompt + '예시: {"severity": "high", "summary": "admin 로그인 실패"} '
prompt = prompt + "로그: 2026-10-06 02:16:03 INFO accepted login for root from 198.51.100.23"
# ── 여기까지 ──

# 1. text 라는 변수를 만들어 call_llm 에 prompt 를 넣어 받은 답을 담으세요
text= call_llm(prompt)

# 2. result 라는 변수를 만들어 parse_llm_json 에 text 를 넣어 받은 것을 담으세요
result= parse_llm_json(text)

# 3. result 가 있으면 severity, "|", summary 를 한 줄로 출력하세요 (if 와 print)
if result:
    print(result["severity"], "ㅣ", result["summary"])
else:
    print("읽지 못했습니다")

IndentationError: unexpected indent (llm_client.py, line 39)

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-6 · 로그 세 줄을 한 번에 분석하기</font></h3></td></tr></table>

`logs` 의 로그 세 줄을 `llm_client.py` 로 분석하고, 한 줄마다 `번호 | 심각도 | 요약` 형식으로 출력하시오. 읽지 못한 로그는 `번호 | 읽지 못했습니다` 로 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `1 | low | lee.yh 사용자 로그인 성공` · `2 | medium | root 계정 로그인 실패` · `3 | high | svc_backup 계정의 /etc/sudoers 접근 권한 거부` 와 비슷한 세 줄 |

**💡 힌트**

1. 질문 문자열은 `instruction + log` 로 만듭니다.
2. `call_llm` 과 `parse_llm_json` 을 차례로 부릅니다. 문제 4-5 와 같은 방법입니다.
3. 몇 번째 로그인지는 반복 전에 `number = 0` 을 두고 반복 안에서 1 씩 더해 셉니다. 한 번 실행에 LLM 을 세 번 호출합니다.

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)          # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
logs = [
    "2026-10-06 09:02:11 INFO accepted login for lee.yh from 192.0.2.17",
    "2026-10-06 02:15:44 WARN failed login for root from 198.51.100.23",
    "2026-10-06 14:31:27 WARN permission denied for svc_backup on /etc/sudoers",
]
instruction = '다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다. 예시: {"severity": "high", "summary": "admin 로그인 실패"} 로그: '
# ── 여기까지 ──

# 1. number 라는 변수를 만들어 0 을 담으세요

# 2. logs 를 하나씩 꺼내 반복하세요. 꺼낸 로그의 변수 이름은 log 입니다

# 3. 반복 안에서 number 에 1 을 더하고, text 에 call_llm(instruction + log) 의 결과를 담으세요

# 4. result 에 parse_llm_json(text) 의 결과를 담으세요

# 5. result 가 있으면 number, "|", severity, "|", summary 를, 없으면 number, "|", "읽지 못했습니다" 를 출력하세요


### 📋 4교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 코드 블록 표시를 지운다 | `text.replace("```json", "").replace("```", "").strip()` 을 세 줄로 나눠 쓴다 |
| 문자열을 딕셔너리로 바꾼다 | `json.loads(clean)` |
| 읽지 못해도 멈추지 않는다 | `try:` … `except json.JSONDecodeError:` … `return None` |
| 질문을 보내 응답 문자열을 받는다 | `call_llm(question)` |
| 응답 문자열을 딕셔너리로 바꾼다 | `parse_llm_json(text)` |
| 꺼내기 전에 확인한다 | `if result:` |

오전 산출물은 `agent_core/llm_client.py` 입니다. 오후에는 이 파일의 두 함수로 AI 에이전트를 만듭니다.

---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 봅니다</mark>

출력 예상 문제의 답은 문제 바로 아래 ✅ 줄에 있습니다.

### 정답 2-2

In [ ]:
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None                                                # 키를 못 찾으면 None 으로 남는다
with open(ENV_PATH, encoding="utf-8") as f:                   # .env 파일을 읽기로 연다
    for line in f:                                            # 파일을 한 줄씩 꺼낸다
        parts = line.strip().split("=", 1)                    # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
        if parts[0] == "GEMINI_API_KEY":                      # = 앞부분이 키 이름이면
            api_key = parts[1]                                # = 뒷부분이 키 값이다
# Gemini 에 요청을 보낼 주소
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
# 요청 헤더 — 키(Bearer 뒤 공백 한 칸)와 본문 형식(JSON)
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# 요청 본문 — 모델 이름과 질문 하나
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
# ── 여기까지 ──

# 1. response 라는 변수를 만들어 url 로 보낸 POST 요청의 결과를 담으세요
#    괄호 안에 url, headers=headers, json=body, timeout=30 을 넣습니다
# POST 요청을 보낸다. 30초 안에 답이 없으면 멈춘다
response = requests.post(url, headers=headers, json=body, timeout=30)

print(response.status_code)

### 정답 2-3

In [ ]:
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None                                                # 키를 못 찾으면 None 으로 남는다
with open(ENV_PATH, encoding="utf-8") as f:                   # .env 파일을 읽기로 연다
    for line in f:                                            # 파일을 한 줄씩 꺼낸다
        parts = line.strip().split("=", 1)                    # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
        if parts[0] == "GEMINI_API_KEY":                      # = 앞부분이 키 이름이면
            api_key = parts[1]                                # = 뒷부분이 키 값이다
# Gemini 에 요청을 보낼 주소
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
# 요청 헤더 — 키(Bearer 뒤 공백 한 칸)와 본문 형식(JSON)
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# 요청 본문 — 모델 이름과 질문 하나
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
# POST 요청을 보낸다. 30초 안에 답이 없으면 멈춘다
response = requests.post(url, headers=headers, json=body, timeout=30)
# ── 여기까지 ──

# 1. data 라는 변수를 만들어 응답을 딕셔너리로 바꾼 것을 담으세요
data = response.json()                                        # 응답(JSON 문자열)을 딕셔너리로 바꾼다

# 2. answer 라는 변수를 만들어 data 에서 꺼낸 답 문장을 담으세요
answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content

print(answer)
print("쓴 토큰:", data["usage"]["total_tokens"])

### 정답 2-4

In [ ]:
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
ip = "211.45.12.9"
question = "IP " + ip + " 는 어느 통신사의 주소입니까? 통신사 이름만 답하세요."
# 요청 본문 — 모델 이름과 질문 하나
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
# 조회 API 에 물어 실제 값을 받는다
truth = requests.get("https://ipwho.is/" + ip, timeout=10).json()
response = post_to_llm(body)                                  # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)
# ── 여기까지 ──

# 1. data 라는 변수를 만들어 응답을 딕셔너리로 바꾼 것을 담으세요
data = response.json()                                        # 응답(JSON 문자열)을 딕셔너리로 바꾼다

answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content

# 2. isp 라는 변수를 만들어 truth 에서 꺼낸 통신사 이름을 담으세요
isp = truth["connection"]["isp"]

print("LLM     :", answer)
print("조회 API :", isp)

### 정답 2-5

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# 요청 본문 — 모델 이름과 질문 하나
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
response = post_to_llm(body, api_key="wrong-key-1234")      # 일부러 틀린 키로 보냅니다
# ── 여기까지 ──

# 1. response 의 상태코드가 200 이면
#    data 라는 변수에 응답을 딕셔너리로 바꿔 담고, 답 문장을 출력하세요
# 2. 200 이 아니면 "실패:" 와 상태코드를 출력하세요
if response.status_code == 200:                               # 200 이면 성공
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    print(data["choices"][0]["message"]["content"])
else:
    print("실패:", response.status_code)

### 정답 2-6

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
logs = [
    "2026-10-06 09:14:52 INFO accepted login for jdoe from 192.0.2.44",
    "2026-10-06 13:07:30 WARN permission denied for guest01 on /etc/shadow",
    "2026-10-06 23:58:11 WARN token expired for deploy from 203.0.113.61",
]
# ── 여기까지 ──

# 1. logs 를 하나씩 꺼내 반복하세요. 꺼낸 로그의 변수 이름은 log 입니다
# 2. 반복 안에서 body 를 만들고, post_to_llm(body) 의 결과를 response 에 담으세요
#    질문은 "한 문장으로 설명하세요. 로그: " + log 입니다
# 3. 상태코드가 200 이면 답 문장을 출력하세요
# 4. 200 이 아니면 "실패:" 와 상태코드를 출력하세요
for log in logs:
    body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": "한 문장으로 설명하세요. 로그: " + log}]}
    response = post_to_llm(body)                              # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)
    if response.status_code == 200:                           # 200 이면 성공
        data = response.json()                                # 응답(JSON 문자열)을 딕셔너리로 바꾼다
        print(data["choices"][0]["message"]["content"])
    else:
        print("실패:", response.status_code)

### 정답 3-2

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
log = "2026-10-06 01:48:09 WARN failed login for root from 198.51.100.23"
# ── 여기까지 ──

# 1. messages 라는 변수를 만들어 system 딕셔너리와 user 딕셔너리를 리스트에 담으세요
#    system 의 content 는 "당신은 보안관제 애널리스트입니다. 한 문장으로만 답합니다." 입니다
messages = [{"role": "system", "content": "당신은 보안관제 애널리스트입니다. 한 문장으로만 답합니다."}, {"role": "user", "content": log}]

# 2. body 라는 변수를 만들어 모델 이름과 messages 를 넣은 본문을 담으세요
# 요청 본문 — 모델 이름과 messages
body = {"model": "gemini-3.5-flash-lite", "messages": messages}

response = post_to_llm(body)                                  # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)

data = response.json()                                        # 응답(JSON 문자열)을 딕셔너리로 바꾼다
answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content
print(answer)

### 정답 3-3

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
log = "2026-10-06 01:48:09 WARN failed login for root from 198.51.100.23"
# ── 여기까지 ──

# 1. question 이라는 변수를 만들어 아래 지시 뒤에 log 를 이어 붙이세요
#    "다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다. 로그: "
question = "다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다. 로그: " + log

# 2. messages 라는 변수를 만들어 system 딕셔너리와 user 딕셔너리를 담으세요
# system 에 역할, user 에 질문
messages = [{"role": "system", "content": "당신은 보안관제 애널리스트입니다."}, {"role": "user", "content": question}]

# 요청 본문 — 모델 이름과 messages
body = {"model": "gemini-3.5-flash-lite", "messages": messages}

response = post_to_llm(body)                                  # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)

data = response.json()                                        # 응답(JSON 문자열)을 딕셔너리로 바꾼다
answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content
print(answer)

### 정답 3-4

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
log = "2026-10-06 01:48:09 WARN failed login for root from 198.51.100.23"
instruction = "다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다."
example = '예시: {"severity": "high", "summary": "admin 로그인 실패"}'
# ── 여기까지 ──

# 1. question 이라는 변수를 만들어 instruction, example, "로그: " + log 를 공백 한 칸씩 두고 이어 붙이세요
question = instruction + " " + example + " 로그: " + log        # 지시 + 예시 + 로그를 이어 붙인다

# 2. messages 라는 변수를 만들어 system 딕셔너리와 user 딕셔너리를 담으세요
# system 에 역할, user 에 질문
messages = [{"role": "system", "content": "당신은 보안관제 애널리스트입니다."}, {"role": "user", "content": question}]

# 요청 본문 — 모델 이름과 messages
body = {"model": "gemini-3.5-flash-lite", "messages": messages}

response = post_to_llm(body)                                  # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)

data = response.json()                                        # 응답(JSON 문자열)을 딕셔너리로 바꾼다
answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content
print(answer)

### 정답 3-5

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
log = "2026-10-06 14:31:27 WARN permission denied for svc_backup on /etc/sudoers"
instruction = "다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다."
example = '예시: {"severity": "high", "summary": "admin 로그인 실패"}'
# ── 여기까지 ──

# 1. question 이라는 변수를 만들어 instruction, example, "로그: " + log 를 이어 붙이세요
question = instruction + " " + example + " 로그: " + log        # 지시 + 예시 + 로그를 이어 붙인다

# 2. messages 라는 변수를 만들어 system 딕셔너리와 user 딕셔너리를 담으세요. system 의 content 는 "당신은 보안관제 애널리스트입니다." 입니다
# system 에 역할, user 에 질문
messages = [{"role": "system", "content": "당신은 보안관제 애널리스트입니다."}, {"role": "user", "content": question}]

# 3. body 라는 변수를 만들어 모델 이름, temperature 0, messages 를 넣은 본문을 담으세요
# temperature 0 — 답이 덜 흔들린다
body = {"model": "gemini-3.5-flash-lite", "temperature": 0, "messages": messages}

response = post_to_llm(body)                                  # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)

data = response.json()                                        # 응답(JSON 문자열)을 딕셔너리로 바꾼다
answer = data["choices"][0]["message"]["content"]             # 답 문장 꺼내기: choices → 0번 → message → content
print(answer)

### 정답 3-6

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
logs = [
    "2026-10-06 09:02:11 INFO accepted login for lee.yh from 192.0.2.17",
    "2026-10-06 02:15:44 WARN failed login for root from 198.51.100.23",
    "2026-10-06 02:16:03 INFO accepted login for root from 198.51.100.23",
]
instruction = "다음 로그의 심각도를 high, medium, low 중 하나로만 답하세요. 로그: "
# ── 여기까지 ──

# 1. number 라는 변수를 만들어 0 을 담으세요
# 2. logs 를 하나씩 꺼내 반복하세요. 꺼낸 로그의 변수 이름은 log 입니다
# 3. 반복 안에서 number 에 1 을 더하고, messages 와 body(temperature 0) 를 만들어 post_to_llm(body) 로 보내세요
# 4. 응답에서 답 단어를 꺼내 answer 에 담으세요
# 5. number, "→", answer 를 출력하세요
number = 0                                                    # 몇 번째 로그인지 센다
for log in logs:
    number = number + 1                                       # 하나 늘린다
    messages = [{"role": "system", "content": "당신은 보안관제 애널리스트입니다."}, {"role": "user", "content": instruction + log}]
    # temperature 0 — 답이 덜 흔들린다
    body = {"model": "gemini-3.5-flash-lite", "temperature": 0, "messages": messages}
    response = post_to_llm(body)                              # 키 읽기 · 헤더 · 요청을 함수가 대신 한다 (2.4)
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    answer = data["choices"][0]["message"]["content"]         # 답 문장 꺼내기: choices → 0번 → message → content
    print(number, "→", answer)

### 정답 4-2

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
text = '```json {"severity": "high", "summary": "root 계정 로그인 실패 4회"} ```'
# ── 여기까지 ──

# 1. clean 이라는 변수를 만들어 text 에서 "```json" 을 지운 문자열을 담으세요
clean = text.replace("```json", "")                           # 코드 블록 표시 ```json 을 지운다

# 2. clean 에서 "```" 을 지운 문자열을 다시 clean 에 담으세요
clean = clean.replace("```", "")                              # 남은 ``` 를 지운다

# 3. clean 의 앞뒤 공백을 지운 문자열을 다시 clean 에 담으세요
clean = clean.strip()                                         # 앞뒤 공백 · 줄바꿈을 지운다

# 4. result 라는 변수를 만들어 clean 을 딕셔너리로 바꾼 것을 담으세요
result = json.loads(clean)

print(result["severity"])
print(result["summary"])

### 정답 4-3

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구


def parse_llm_json(text):                                     # LLM 의 답을 딕셔너리로 바꾸는 함수. 못 읽으면 None
    clean = text.replace("```json", "")                       # 코드 블록 표시 ```json 을 지운다
    clean = clean.replace("```", "")                          # 남은 ``` 를 지운다
    clean = clean.strip()                                     # 앞뒤 공백 · 줄바꿈을 지운다

    try:                                                      # 아래 줄을 해 본다
        # 1. clean 을 딕셔너리로 바꾼 것을 return 하세요
        return json.loads(clean)                              # 문자열을 딕셔너리로 바꿔 돌려준다
    except json.JSONDecodeError:                              # JSON 으로 읽지 못하면 여기로 온다
        # 2. None 을 return 하세요
        return None


print(parse_llm_json('```json{"severity": "low"}```'))
print(parse_llm_json("분석할 수 없습니다."))

### 정답 4-4

In [ ]:
%%writefile llm_client.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import os                                                     # 폴더 · 파일을 다루는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# Gemini 에 요청을 보낼 주소
URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL = "gemini-3.5-flash-lite"                               # 쓸 모델 이름


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                          # 지금 폴더에서 다섯 칸 위까지 찾습니다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 돌려준다
        folder = os.path.dirname(folder)        # 한 칸 위 폴더
    return None


def read_api_key():                                           # .env 에서 키를 읽어 돌려주는 함수
    api_key = None                                            # 키를 못 찾으면 None 으로 남는다
    with open(find_env(), encoding="utf-8") as f:             # .env 를 찾아 읽기로 연다
        for line in f:                                        # 파일을 한 줄씩 꺼낸다
            parts = line.strip().split("=", 1)                # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
            if parts[0] == "GEMINI_API_KEY":                  # = 앞부분이 키 이름이면
                api_key = parts[1]                            # = 뒷부분이 키 값이다
    return api_key                                            # 읽은 키를 돌려준다


def call_llm(question):                                       # 질문을 받아 답 문장을 돌려주는 함수
    # 요청 헤더 — .env 에서 읽은 키를 넣는다
    headers = {"Authorization": "Bearer " + read_api_key(), "Content-Type": "application/json"}
    # 1. body 라는 변수를 만들어 MODEL, temperature 0, question 을 넣은 본문을 담으세요
    # 요청 본문 — 모델 · temperature 0 · 질문
    body = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": question}]}

    # 2. response 라는 변수를 만들어 URL 로 보낸 POST 요청의 결과를 담으세요
    # POST 요청을 보낸다. 30초 안에 답이 없으면 멈춘다
    response = requests.post(URL, headers=headers, json=body, timeout=30)

    if response.status_code != 200:                           # 200(성공)이 아니면
        print("[LLM 호출 실패] 상태 코드", response.status_code)      # 429 면 1분 기다린 뒤 다시 실행
        return ""                                             # 실패하면 빈 문자열을 돌려준다
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    return data["choices"][0]["message"]["content"]           # 답 문장만 돌려준다


def parse_llm_json(text):                                     # LLM 의 답을 딕셔너리로 바꾸는 함수. 못 읽으면 None
    clean = text.replace("```json", "")                       # 코드 블록 표시 ```json 을 지운다
    clean = clean.replace("```", "")                          # 남은 ``` 를 지운다
    clean = clean.strip()                                     # 앞뒤 공백 · 줄바꿈을 지운다

    try:                                                      # 아래 줄을 해 본다
        return json.loads(clean)                              # 문자열을 딕셔너리로 바꿔 돌려준다
    except json.JSONDecodeError:                              # JSON 으로 읽지 못하면 여기로 온다
        return None

### 정답 4-5

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)          # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
prompt = '다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다. '
prompt = prompt + '예시: {"severity": "high", "summary": "admin 로그인 실패"} '
prompt = prompt + "로그: 2026-10-06 02:16:03 INFO accepted login for root from 198.51.100.23"
# ── 여기까지 ──

# 1. text 라는 변수를 만들어 call_llm 에 prompt 를 넣어 받은 답을 담으세요
text = call_llm(prompt)                                       # LLM 에 보내 답 문장을 받는다

# 2. result 라는 변수를 만들어 parse_llm_json 에 text 를 넣어 받은 것을 담으세요
result = parse_llm_json(text)                                 # 답을 딕셔너리로 바꾼다. 못 읽으면 None

# 3. result 가 있으면 severity, "|", summary 를 한 줄로 출력하세요 (if 와 print)
if result:                                                    # None 이 아니면 (답을 읽었으면)
    print(result["severity"], "|", result["summary"])
else:
    print("읽지 못했습니다")

### 정답 4-6

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)          # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
logs = [
    "2026-10-06 09:02:11 INFO accepted login for lee.yh from 192.0.2.17",
    "2026-10-06 02:15:44 WARN failed login for root from 198.51.100.23",
    "2026-10-06 14:31:27 WARN permission denied for svc_backup on /etc/sudoers",
]
instruction = '다음 로그를 분석해 JSON 으로만 답하세요. 키는 severity 와 summary 둘입니다. severity 는 high, medium, low 중 하나입니다. 예시: {"severity": "high", "summary": "admin 로그인 실패"} 로그: '
# ── 여기까지 ──

# 1. number 라는 변수를 만들어 0 을 담으세요
# 2. logs 를 하나씩 꺼내 반복하세요. 꺼낸 로그의 변수 이름은 log 입니다
# 3. 반복 안에서 number 에 1 을 더하고, text 에 call_llm(instruction + log) 의 결과를 담으세요
# 4. result 에 parse_llm_json(text) 의 결과를 담으세요
# 5. result 가 있으면 number, "|", severity, "|", summary 를, 없으면 number, "|", "읽지 못했습니다" 를 출력하세요
number = 0                                                    # 몇 번째 로그인지 센다
for log in logs:
    number = number + 1                                       # 하나 늘린다
    text = call_llm(instruction + log)
    result = parse_llm_json(text)                             # 답을 딕셔너리로 바꾼다. 못 읽으면 None
    if result:                                                # None 이 아니면 (답을 읽었으면)
        print(number, "|", result["severity"], "|", result["summary"])
    else:
        print(number, "|", "읽지 못했습니다")